# 충남대학교 학내 정보 Q&A 시스템

**자연어처리 텀프로젝트 — 202202497 장윤상**

RAG 기반 학내 정보 질의응답 시스템

| 구성 요소 | 선택 |
|-----------|------|
| Base 모델 | `google/gemma-4-12b-it` (4bit NF4) |
| 파인튜닝 | 없음 (base 모델 직접 사용) |
| 임베딩 | `BAAI/bge-m3` |
| 벡터 DB | ChromaDB |

> **사용법**: 런타임 → GPU(T4) 선택 후 **"모두 실행"**

## 0. 소스코드 클론 & 의존성 설치

설치 완료 후 **런타임이 자동 재시작**됩니다. 재시작 후 다시 "모두 실행"하면 됩니다.

In [ ]:
import os

# 1) 소스코드 클론
REPO_URL = "https://github.com/adoveflash/cnu_qa_system.git"
REPO_DIR = "cnu_qa_system"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"이미 존재: {REPO_DIR}")

# 디렉터리 이동 (중복 실행 방지)
if not os.getcwd().endswith(REPO_DIR):
    os.chdir(REPO_DIR)
print(f"작업 디렉터리: {os.getcwd()}")

# 2) 의존성 설치 — transformers만 최신 (Gemma 4 지원)
#    torch, bitsandbytes는 Colab 기본 유지 (CUDA 12 호환)
!pip install -q --upgrade transformers
!pip install -q sentence-transformers chromadb gradio accelerate \
    requests beautifulsoup4 pdfplumber huggingface_hub

# 3) 런타임 재시작 (transformers 업그레이드 반영)
#    재시작 후 이 셀은 스킵되고 다음 셀부터 실행됨
try:
    import transformers
    if not hasattr(transformers, 'Gemma4ForCausalLM'):
        print("런타임 재시작 필요 — 자동 재시작합니다...")
        os.kill(os.getpid(), 9)
    else:
        print(f"transformers {transformers.__version__} — Gemma 4 지원 확인")
except:
    pass

## 1. 환경 설정

In [ ]:
import os
import json
import time
import torch
import random

# 재시작 후 디렉터리 복원
REPO_DIR = "cnu_qa_system"
if os.path.exists(REPO_DIR) and not os.getcwd().endswith(REPO_DIR):
    os.chdir(REPO_DIR)

SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

print(f"작업 디렉터리: {os.getcwd()}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. 데이터 다운로드

벡터 DB 인덱스와 코퍼스를 HuggingFace Hub에서 다운로드합니다.

In [ ]:
from huggingface_hub import snapshot_download

HF_REPO = "adoveflash/cnu-qa-system"

# 벡터 DB 인덱스 다운로드
VECTOR_DB_PATH = "data/vector_db"
if not os.path.exists(VECTOR_DB_PATH):
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=".",
        allow_patterns=["data/vector_db/**"],
    )
    print(f"벡터 DB 다운로드 완료: {VECTOR_DB_PATH}")
else:
    print(f"벡터 DB 이미 존재: {VECTOR_DB_PATH}")

# 코퍼스 (chunks) 다운로드
CHUNKS_PATH = "data/corpus/chunks.jsonl"
if not os.path.exists(CHUNKS_PATH):
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=".",
        allow_patterns=["data/corpus/chunks.jsonl"],
    )
    print(f"코퍼스 다운로드 완료: {CHUNKS_PATH}")
else:
    print(f"코퍼스 이미 존재: {CHUNKS_PATH}")

## 3. 모델 로드

Gemma 4 12B (4bit NF4 양자화)를 로드합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "google/gemma-4-12b-it"

# 4bit NF4 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 토크나이저
print("[1/2] 토크나이저 로드")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델
print("[2/2] 모델 로드 (4bit NF4)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

vram_gb = torch.cuda.memory_reserved() / 1024**3
print(f"\n모델 로드 완료 — VRAM 사용: {vram_gb:.2f} GB")

from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

# 임베딩 모델 (GPU)
print("임베딩 모델 로드: BAAI/bge-m3")
embed_model = SentenceTransformer("BAAI/bge-m3")

# ChromaDB
db_path = Path(VECTOR_DB_PATH)
client = chromadb.PersistentClient(path=str(db_path))
collection = client.get_collection("cnu_chunks")
print(f"벡터 DB 로드 완료: {collection.count()}개 청크")


def retrieve(query, top_k=5):
    """질문과 유사한 청크를 검색한다."""
    query_emb = embed_model.encode([query]).tolist()[0]
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return results


def build_context(query, top_k=5):
    """질문에 대한 컨텍스트 문자열과 출처 URL을 생성한다."""
    results = retrieve(query, top_k)
    context_parts = []
    urls = []
    for i in range(len(results["ids"][0])):
        title = results["metadatas"][0][i]["title"]
        text = results["documents"][0][i]
        url = results["metadatas"][0][i]["url"]
        context_parts.append(f"[참고{i+1}] {title}\n{text}")
        if url and url not in urls:
            urls.append(url)
    return "\n\n".join(context_parts), urls


# 검색 테스트
test_results = retrieve("졸업 요건이 어떻게 되나요?", top_k=3)
for i, doc in enumerate(test_results["documents"][0]):
    dist = test_results["distances"][0][i]
    print(f"  [{i+1}] (dist={dist:.3f}) {doc[:80]}...")

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

# 임베딩 모델 (CPU에 로드 → GPU VRAM 절약)
print("임베딩 모델 로드: BAAI/bge-m3 (CPU)")
embed_model = SentenceTransformer("BAAI/bge-m3", device="cpu")

# ChromaDB
db_path = Path(VECTOR_DB_PATH)
client = chromadb.PersistentClient(path=str(db_path))
collection = client.get_collection("cnu_chunks")
print(f"벡터 DB 로드 완료: {collection.count()}개 청크")


def retrieve(query, top_k=5):
    """질문과 유사한 청크를 검색한다."""
    query_emb = embed_model.encode([query]).tolist()[0]
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return results


def build_context(query, top_k=5):
    """질문에 대한 컨텍스트 문자열과 출처 URL을 생성한다."""
    results = retrieve(query, top_k)
    context_parts = []
    urls = []
    for i in range(len(results["ids"][0])):
        title = results["metadatas"][0][i]["title"]
        text = results["documents"][0][i]
        url = results["metadatas"][0][i]["url"]
        context_parts.append(f"[참고{i+1}] {title}\n{text}")
        if url and url not in urls:
            urls.append(url)
    return "\n\n".join(context_parts), urls


# 검색 테스트
test_results = retrieve("졸업 요건이 어떻게 되나요?", top_k=3)
for i, doc in enumerate(test_results["documents"][0]):
    dist = test_results["distances"][0][i]
    print(f"  [{i+1}] (dist={dist:.3f}) {doc[:80]}...")

## 5. 추론 함수 정의

In [ ]:
from datetime import datetime, timezone, timedelta

KST = timezone(timedelta(hours=9))
now = datetime.now(KST)
today = now.strftime("%Y-%m-%d (%A)")
month = now.month
if 3 <= month <= 8:
    semester = f"{now.year}학년도 1학기"
else:
    year = now.year if month >= 9 else now.year - 1
    semester = f"{year}학년도 2학기"

SYSTEM_PROMPT = (
    f"너는 충남대학교 학내 정보를 안내하는 친절한 AI 챗봇이야.\n"
    f"오늘 날짜: {today} | 현재 학기: {semester}\n\n"
    "대화 스타일:\n"
    "- 친근하고 자연스러운 말투로 대답해. 딱딱하지 않게, 친구에게 설명하듯이.\n"
    "- '~해요', '~이에요' 같은 존댓말을 사용하되 부드럽게.\n"
    "- 질문에 맞는 핵심 정보를 먼저 알려주고, 필요하면 추가 설명을 덧붙여.\n\n"
    "규칙:\n"
    "1. 주어진 참고 자료에 있는 정보를 기반으로 답변해.\n"
    "2. 참고 자료에 없는 내용은 절대 지어내지 마. '해당 정보를 찾지 못했어요'라고 솔직히 답해.\n"
    "3. 메뉴명, 일정명 등 고유명사에 형용사나 수식어를 추가하지 마.\n"
    "4. 반드시 한국어로만 답변해."
)

# 출처 라벨 매핑
_SOURCE_LABELS = {
    "computer.cnu.ac.kr": "컴퓨터융합학부",
    "plus.cnu.ac.kr": "충남대 공식",
    "job.cnu.ac.kr": "인재개발원",
    "sugang.cnu.ac.kr": "수강신청",
    "www.cnucoop.co.kr": "생활협동조합",
    "mobileadmin.cnu.ac.kr": "충남대 식단",
}


def _format_sources(urls):
    """URL 리스트를 출처 텍스트로 변환한다."""
    if not urls:
        return ""
    from urllib.parse import urlparse
    seen = []
    for url in urls:
        for domain, label in _SOURCE_LABELS.items():
            if domain in url:
                if label not in seen:
                    seen.append(label)
                break
        else:
            host = urlparse(url).hostname or url
            short = host.replace("www.", "").split(".")[0]
            if short not in seen:
                seen.append(short)
    return "\n\n출처: " + ", ".join(seen)


def generate_answer(question, max_new_tokens=1024):
    """RAG 검색 → LLM 답변 생성 파이프라인."""
    context, urls = build_context(question)
    if not context:
        return "관련 정보를 찾을 수 없습니다."

    user_msg = (
        f"아래 참고 자료를 반드시 읽고, 참고 자료에 있는 내용만으로 답변해.\n\n"
        f"참고 자료:\n{context}\n\n질문: {question}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    torch.manual_seed(SEED)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer += _format_sources(urls)
    return answer


# 추론 테스트
print(generate_answer("수강신청은 어떻게 하나요?"))

## 6. 배치 추론

In [ ]:
# Task 2: chat_output.json
CHAT_TEST = "data/test_chat.json"
CHAT_OUTPUT = "outputs/chat_output.json"

if os.path.exists(CHAT_TEST):
    with open(CHAT_TEST, encoding="utf-8") as f:
        chat_data = json.load(f)

    os.makedirs("outputs", exist_ok=True)
    results = []
    for i, item in enumerate(chat_data, 1):
        q = item["user"]
        print(f"[{i}/{len(chat_data)}] {q[:50]}...", end=" ", flush=True)
        start = time.time()
        answer = generate_answer(q)
        elapsed = time.time() - start
        results.append({"user": q, "model": answer})
        print(f"{elapsed:.1f}s")

    with open(CHAT_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n저장 완료: {CHAT_OUTPUT} ({len(results)}건)")
else:
    print(f"{CHAT_TEST} 파일 없음 — 건너뜀")

In [ ]:
# Task 3: realtime_output.json
RT_TEST = "data/test_realtime.json"
RT_OUTPUT = "outputs/realtime_output.json"

if os.path.exists(RT_TEST):
    with open(RT_TEST, encoding="utf-8") as f:
        rt_data = json.load(f)

    os.makedirs("outputs", exist_ok=True)
    results = []
    for i, item in enumerate(rt_data, 1):
        q = item["user"]
        print(f"[{i}/{len(rt_data)}] {q[:50]}...", end=" ", flush=True)
        start = time.time()
        answer = generate_answer(q)
        elapsed = time.time() - start
        results.append({"user": q, "model": answer})
        print(f"{elapsed:.1f}s")

    with open(RT_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\n저장 완료: {RT_OUTPUT} ({len(results)}건)")
else:
    print(f"{RT_TEST} 파일 없음 — 건너뜀")

## 7. Gradio UI 실행

In [ ]:
import gradio as gr

with gr.Blocks(title="충남대학교 학내 정보 Q&A") as demo:
    gr.Markdown(
        "# 충남대학교 학내 정보 Q&A 시스템\n"
        "학사, 장학금, 취업, 식단, 셔틀버스 등을 질문해보세요."
    )
    with gr.Row():
        question_input = gr.Textbox(
            label="질문",
            placeholder="예: 졸업 요건이 어떻게 되나요?",
            lines=2,
        )
    submit_btn = gr.Button("질문하기", variant="primary")
    answer_output = gr.Textbox(label="답변", lines=10, interactive=False)

    gr.Examples(
        examples=[
            "컴퓨터융합학부 졸업 요건이 어떻게 되나요?",
            "수강신청은 언제 하나요?",
            "제1학생회관 점심 메뉴 알려줘",
            "셔틀버스 시간표 알려줘",
            "장학금 신청은 어떻게 하나요?",
        ],
        inputs=question_input,
    )

    submit_btn.click(fn=generate_answer, inputs=question_input, outputs=answer_output)
    question_input.submit(fn=generate_answer, inputs=question_input, outputs=answer_output)

demo.launch(share=True)

## 8. 결과 확인

In [ ]:
# 생성된 결과 파일 확인
for path in ["outputs/chat_output.json", "outputs/realtime_output.json"]:
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"\n{'='*60}")
        print(f"{path} — {len(data)}건")
        print(f"{'='*60}")
        for item in data[:3]:
            print(f"\nQ: {item['user'][:80]}")
            print(f"A: {item['model'][:200]}...")
    else:
        print(f"{path} — 파일 없음")